In [6]:
# load module
import os
#os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import pytorch_lightning as pl
import seaborn as sns
from scipy.signal import find_peaks

from my_funs import *
from ewma import  data_processing
from sklearn import metrics
from matplotlib import gridspec
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger


from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet , EncoderNormalizer , GroupNormalizer
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

#import tensorflow as tf 
import tensorboard as tb 
#tf.io.gfile = tb.compat.tensorflow_stub.io.gfile

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'

dongs = ['강남동', '교  동', '근화동', '남  면', '남산면', '동  면', '동내면', '동산면', 
         '북산면','사북면', '서  면', '석사동', '소양동', '신동면', '신북읍', '신사우동', '약사명동', '조운동','퇴계동', 
         '효자1동', '후평1동']

In [14]:
def moving_average_alpha_both(df: pd.DataFrame, unit: float):
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        max_value = dong_df['count'].max()
        back_ewma = dong_df['count'].ewm(alpha = unit).mean()

        inv_dong_df = dong_df[::-1]
        for_ewma = inv_dong_df['count'].ewm(alpha = unit).mean()
        
        ewma = (for_ewma + back_ewma) / 2
        ewma = ewma / ewma.max() * max_value
        df['count'][dong_df.index] = ewma 
    return df

def moving_average_alpha_noise(df: pd.DataFrame, unit: float):
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        max_value = dong_df['count'].max()
        back_ewma = dong_df['count'].ewm(alpha = unit).mean()

        inv_dong_df = dong_df[::-1]
        for_ewma = inv_dong_df['count'].ewm(alpha = unit).mean()
        
        ewma = (for_ewma + back_ewma) / 2
        ewma = ewma / ewma.max() * max_value
        df['count'][dong_df.index] = ewma 
    df['count'] += np.random.normal(0.02,0.04, len(df))
    return df

def moving_average_hour(df: pd.DataFrame, unit = 0):
    for dong in df['h_dong'].unique():
        #print(dong)
        dong_df = df[df['h_dong'] == dong]
        over0 = dong_df[dong_df['count'] > 0].index
        for idx in over0:
            #print(idx , df['time_idx'].loc[idx])
            value = df['count'].loc[idx] 
            try:
                df['count'].loc[idx-21] += value / 2
                df['count'].loc[idx+21] += value / 2
            except:
                pass
    return df



In [100]:
test_data = data_processing('../../test.csv' , 0.9999999999999999, moving_average_alpha_noise)
test_data[test_data['count'] >= 0]

,REG_DTIME,h_dong,count,pops,windspd,humid,temp,precip_form,precip,isHoliday,DOW,HOD,MOY,time_idx
1,2022-01-01 00:00:00,신동면,0.007327,2602.0,0.3,48.0,-8.6,0.0,0.0,True,5,0,1,0
2,2022-01-01 00:00:00,강남동,0.072151,22607.0,1.4,60.0,-9.7,0.0,0.0,True,5,0,1,0
6,2022-01-01 00:00:00,동내면,0.085129,17202.0,1.4,60.0,-9.7,0.0,0.0,True,5,0,1,0
7,2022-01-01 00:00:00,효자1동,0.057135,4413.0,1.4,60.0,-9.7,0.0,0.0,True,5,0,1,0
8,2022-01-01 00:00:00,남산면,0.004279,3428.0,0.3,48.0,-8.6,0.0,0.0,True,5,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29225,2022-02-27 23:00:00,동 면,0.027458,19522.0,0.7,80.0,0.8,0.0,0.0,False,6,23,2,1391
29226,2022-02-27 23:00:00,후평1동,0.027804,11511.0,0.7,80.0,0.8,0.0,0.0,False,6,23,2,1391
29228,2022-02-27 23:00:00,북산면,0.074195,965.0,0.0,84.0,2.6,0.0,0.0,False,6,23,2,1391
29230,2022-02-27 23:00:00,동산면,0.018570,1445.0,0.5,89.0,1.6,0.0,0.0,False,6,23,2,1391


# ewma 한계

ewma는 지수적으로 감소하므로 0보다 살짝 높은 값이 살짝 남는다.  
1시간 추가하는 식으로 control하기 어려움   

In [103]:
test_data = data_processing('../../test.csv' , 0, moving_average_alpha_both)
dong_data = test_data[test_data['h_dong']=='강남동']
dong_data.reset_index(inplace=True, drop=True)
len(dong_data[dong_data['count'] > 0])

37

In [21]:
test_data = data_processing('../../test.csv' , 0.9999999999999999, moving_average_alpha_both)
dong_data = test_data[test_data['h_dong']=='강남동']
dong_data.reset_index(inplace=True, drop=True)
len(dong_data[dong_data['count'] > 0])

815

# hour processing

실제 발생 시간이 idx에서 나타났다면 idx-1 , idx+1에 idx의 값 / 2를 양 옆에 넣어줌  

In [22]:
# hour processing 적용 X
test_data = data_processing('../../train.csv' , 0, None)
dong_data = test_data[test_data['h_dong']=='강남동']
dong_data.reset_index(inplace=True, drop=True)
dong_data[dong_data['count'] > 0]

,REG_DTIME,h_dong,count,pops,windspd,humid,temp,precip_form,precip,isHoliday,DOW,HOD,MOY,time_idx
32,2021-01-02 08:00:00,강남동,1,19089.0,2.6,28.0,-2.8,0.0,0.0,False,5,8,1,32
85,2021-01-04 13:00:00,강남동,1,19089.0,0.5,88.0,-2.6,3.0,0.4,False,0,13,1,85
199,2021-01-09 07:00:00,강남동,1,19089.0,2.7,36.0,-8.4,0.0,0.0,False,5,7,1,199
203,2021-01-09 11:00:00,강남동,1,19089.0,0.7,55.0,-12.5,0.0,0.0,False,5,11,1,203
213,2021-01-09 21:00:00,강남동,1,19089.0,0.0,80.0,-18.0,0.0,0.0,False,5,21,1,213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8530,2021-12-22 10:00:00,강남동,2,22437.0,0.1,74.0,-0.6,0.0,0.0,False,2,10,12,8530
8625,2021-12-26 09:00:00,강남동,1,22437.0,2.1,34.0,-9.4,0.0,0.0,False,6,9,12,8625
8632,2021-12-26 16:00:00,강남동,2,22437.0,1.1,57.0,-13.8,0.0,0.0,False,6,16,12,8632
8682,2021-12-28 18:00:00,강남동,1,22437.0,0.3,88.0,-5.7,0.0,0.0,False,1,18,12,8682


In [15]:
# hour preocessin 적용 
test_data = data_processing('../../train.csv' , 99, moving_average_hour)
dong_data = test_data[test_data['h_dong']=='강남동']
dong_data.reset_index(inplace=True, drop=True)
dong_data[dong_data['count'] > 0]

,REG_DTIME,h_dong,count,pops,windspd,humid,temp,precip_form,precip,isHoliday,DOW,HOD,MOY,time_idx
31,2021-01-02 07:00:00,강남동,0.5,19089.0,2.2,27.0,-2.0,0.0,0.0,False,5,7,1,31
32,2021-01-02 08:00:00,강남동,1.0,19089.0,2.6,28.0,-2.8,0.0,0.0,False,5,8,1,32
33,2021-01-02 09:00:00,강남동,0.5,19089.0,2.1,29.0,-3.3,0.0,0.0,False,5,9,1,33
84,2021-01-04 12:00:00,강남동,0.5,19089.0,2.7,71.0,-1.8,3.0,0.0,False,0,12,1,84
85,2021-01-04 13:00:00,강남동,1.0,19089.0,0.5,88.0,-2.6,3.0,0.4,False,0,13,1,85
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8682,2021-12-28 18:00:00,강남동,1.0,22437.0,0.3,88.0,-5.7,0.0,0.0,False,1,18,12,8682
8683,2021-12-28 19:00:00,강남동,0.5,22437.0,0.3,87.0,-5.1,0.0,0.0,False,1,19,12,8683
8755,2021-12-31 19:00:00,강남동,0.5,22437.0,0.7,78.0,-14.3,0.0,0.0,False,4,19,12,8755
8756,2021-12-31 20:00:00,강남동,1.0,22437.0,0.3,83.0,-14.8,0.0,0.0,False,4,20,12,8756
